# Invoice & Receipt Processing — Colab Training Notebook ★ H100 AUTOPILOT, RESUME-SAFE, FLEXIBLE-GPU

Trains the **Invoice & Receipt Processing (Document-AI / LayoutLMv3 KIE)** system (Project #4) on Google Colab.

✅ **One button** = fine-tune LayoutLMv3 (resume-safe) → evaluate (seqeval entity-F1) → benchmark → error analysis → **auto-generate `report.pdf` + `slides.pptx`**
✅ **Survives disconnects**: artifacts + checkpoints on Drive; re-run → training **resumes**.
✅ **Flexible GPU**: auto-detects **H100 / A100 / L4 / T4** → batch size + precision.
✅ **Colab-safe deps**: never reinstalls torch; installs Tesseract + Poppler via apt.

> Set `GIT_REPO_URL` in **Controls**, then `Runtime → Run all`.


In [ ]:
#@title 0) Controls  { display-mode: "form" }
GIT_REPO_URL = "https://github.com/ledinhminhquan/04_Invoice_Receipt_Processing"  #@param {type:"string"}USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_PROJECT_DIR = "NLP_Project/invoice_ai"  #@param {type:"string"}
DRIVE_REPO_DIR = ""  #@param {type:"string"}

# MODEL: layoutlmv3-base (accuracy, non-commercial) or LiLT (MIT, commercial)
LAYOUT_MODEL = "microsoft/layoutlmv3-base"  #@param ["microsoft/layoutlmv3-base", "SCUT-DLVCLab/lilt-roberta-en-base", "microsoft/layoutlmv3-large"]
DATASET = "sroie"  #@param ["sroie", "funsd"]
NUM_EPOCHS = 8  #@param {type:"integer"}

RUN_AUTOPILOT = True  #@param {type:"boolean"}
DEBUG_LIMIT = 0       #@param {type:"integer"}

PROJECT_TITLE = "Invoice & Receipt Processing System"  #@param {type:"string"}
AUTHOR = "Le Dinh Minh Quan"  #@param {type:"string"}
print("Controls set. model:", LAYOUT_MODEL, "| dataset:", DATASET)


In [ ]:
#@title 1) Check GPU
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
#@title 2) Install system OCR/PDF tools (Tesseract + Poppler)
!apt-get -qq update && apt-get -qq install -y tesseract-ocr poppler-utils >/dev/null 2>&1
print("[OK] tesseract + poppler installed")


In [ ]:
#@title 3) Mount Drive + artifact paths & HF caches (before importing torch)
import os
from pathlib import Path
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
else:
    BASE = Path("/content/invoice_ai_artifacts")
ART = BASE / "artifacts"
for sub in ("data", "models", "runs"):
    (ART / sub).mkdir(parents=True, exist_ok=True)
(BASE / "hf_cache").mkdir(parents=True, exist_ok=True)
os.environ["INVOICE_AI_ARTIFACTS_DIR"] = str(ART)
os.environ["INVOICE_AI_MODEL_DIR"]     = str(ART / "models")
os.environ["INVOICE_AI_RUN_DIR"]       = str(ART / "runs")
os.environ["INVOICE_AI_DATA_DIR"]      = str(ART / "data")
os.environ["HF_HOME"]                  = str(BASE / "hf_cache")
print("Artifacts ->", ART)


In [ ]:
#@title 4) Get the project source (git clone, or copy from Drive)
import os, shutil, subprocess
from pathlib import Path
REPO = Path("/content/invoice_ai_repo")
def _ok(p): return (p/"pyproject.toml").exists() and (p/"src"/"invoice_ai").exists()
if _ok(REPO): print("[OK] repo present")
elif GIT_REPO_URL.strip():
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.run(["git","clone","--depth","1",GIT_REPO_URL.strip(),str(REPO)], check=True)
elif DRIVE_REPO_DIR.strip():
    shutil.copytree(Path("/content/drive/MyDrive")/DRIVE_REPO_DIR.strip(), REPO, dirs_exist_ok=True)
else:
    raise SystemExit("Set GIT_REPO_URL (recommended) or DRIVE_REPO_DIR in Controls.")
assert _ok(REPO); os.chdir(REPO); print("[OK] cwd:", os.getcwd())


In [ ]:
#@title 5) Install dependencies (Colab-safe: never touch torch)
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements_colab.txt"], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",".","--no-deps"], check=True)
print("[OK] deps installed (torch untouched)")


In [ ]:
#@title 6) Verify env + performance knobs
import torch, transformers, datasets
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("[WARN] No GPU — use Runtime -> Change runtime type -> GPU.")


In [ ]:
#@title 7) Auto GPU profile (batch size + precision)
import torch
prof = {"batch": 4, "bf16": False, "fp16": True}
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0).upper(); bf16 = torch.cuda.is_bf16_supported()
    if "H100" in name or "A100" in name: b = 16
    elif "L4" in name or "L40" in name: b = 8
    elif "T4" in name or "V100" in name: b = 4
    else: b = 8
    if "large" in LAYOUT_MODEL.lower(): b = max(2, b // 2)
    prof = {"batch": b, "bf16": bool(bf16), "fp16": (not bf16)}
print("GPU profile:", prof)


In [ ]:
#@title 8) Write the Colab training config (configs/train_colab.yaml)
import yaml
from pathlib import Path
cfg = yaml.safe_load(Path("configs/train.yaml").read_text(encoding="utf-8"))
cfg["project_title"] = PROJECT_TITLE; cfg["author"] = AUTHOR
cfg["model"]["layout_model"] = LAYOUT_MODEL
cfg["model"]["num_train_epochs"] = int(NUM_EPOCHS)
cfg["model"]["per_device_train_batch_size"] = prof["batch"]
cfg["model"]["bf16"] = prof["bf16"]; cfg["model"]["fp16"] = prof["fp16"]
Path("configs/train_colab.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
print("Wrote configs/train_colab.yaml | model:", LAYOUT_MODEL, "| batch:", prof["batch"])


In [ ]:
#@title 9) Dataset sanity check (download + label set)
import os
os.environ["INVOICE_AI_INFER_CONFIG"] = "configs/train_colab.yaml"
from invoice_ai.config import load_config
from invoice_ai.data.dataset import load_kie_dataset
cfg = load_config("configs/train_colab.yaml")
limit = DEBUG_LIMIT if DEBUG_LIMIT>0 else None
splits, id2label, _, words_col = load_kie_dataset(cfg.data, which=DATASET, limit=limit)
print("labels:", list(id2label.values()))
print("train/val/test:", len(splits["train"]), len(splits["validation"]), len(splits["test"]) if splits["test"] else 0)


In [ ]:
#@title 10) ★ ONE BUTTON: autopilot (resume-safe). Re-run after a disconnect to RESUME.
import subprocess, sys
if RUN_AUTOPILOT:
    cmd = [sys.executable,"-m","invoice_ai.cli","autopilot","--config","configs/train_colab.yaml",
           "--title",PROJECT_TITLE,"--author",AUTHOR]
    if DEBUG_LIMIT>0: cmd += ["--limit", str(DEBUG_LIMIT)]
    print("Running:", " ".join(cmd)); subprocess.run(cmd, check=False)
else:
    print("RUN_AUTOPILOT off — use the individual cells below.")


## Individual steps (optional) — idempotent + resume-safe

In [ ]:
#@title 11a) Fine-tune LayoutLMv3 (resumes from last checkpoint on Drive)
!python -m invoice_ai.cli train --config configs/train_colab.yaml --dataset {DATASET} {("--limit " + str(DEBUG_LIMIT)) if DEBUG_LIMIT>0 else ""}


In [ ]:
#@title 11b) Evaluate (entity-F1 + end-to-end agent)
!python -m invoice_ai.cli evaluate --config configs/train_colab.yaml --dataset {DATASET}


In [ ]:
#@title 12) Diagnostics: eval metrics + model metadata
import json, os
from pathlib import Path
run_dir = Path(os.environ["INVOICE_AI_RUN_DIR"])
evals = sorted(run_dir.glob("eval-*/eval.json"))
if evals: print("== eval =="); print(json.dumps(json.loads(evals[-1].read_text()).get("summary", {}), indent=2))
meta = Path(os.environ["INVOICE_AI_MODEL_DIR"]) / "layout_extractor" / "latest" / "model_metadata.json"
if meta.exists():
    m = json.loads(meta.read_text()); print("\nmodel:", m.get("base_model"), "| version:", m.get("version"), "| val:", m.get("metrics", {}).get("val", {}).get("eval_f1"))


## ✅ Test the trained model

In [ ]:
#@title 13a) Run the validation agent on built-in invoices (offline; shows reconciliation)
!python -m invoice_ai.cli demo-agent --config configs/train_colab.yaml


In [ ]:
#@title 13b) Run the fine-tuned LayoutLMv3 extractor on a real test document
from invoice_ai.config import load_config
from invoice_ai.models.layout_extractor import LayoutExtractor
from invoice_ai.data.dataset import load_kie_dataset
cfg = load_config("configs/train_colab.yaml")
splits, id2label, _, words_col = load_kie_dataset(cfg.data, which=DATASET)
test = splits["test"] or splits["validation"]
ex = test[0]
ext = LayoutExtractor.from_config(cfg.model)
tokens = [{"text": w, "bbox": b, "conf": 1.0, "page": 1} for w, b in zip(ex[words_col], ex["bboxes"])]
fields = ext.extract(ex["image"].convert("RGB"), tokens)
for k, fv in fields.items():
    print(f"{k:16s} = {fv.value!r:30s} (conf {fv.confidence:.2f})")


In [ ]:
#@title 14) Locate the deliverables (report.pdf + slides.pptx + bundle)
import os
from pathlib import Path
sub = Path(os.environ["INVOICE_AI_ARTIFACTS_DIR"]) / "submission"
latest = sorted(sub.glob("submission-*"))
if latest:
    print("Submission folder:", latest[-1])
    for f in sorted(latest[-1].iterdir()):
        print("  -", f.name, f"({f.stat().st_size//1024} KB)" if f.is_file() else "")
else:
    print("Run the autopilot cell (10) first.")


In [ ]:
#@title 15) (Optional) Serve the API + demo (port 7860)
RUN_SERVER = False  #@param {type:"boolean"}
if RUN_SERVER:
    import os; os.environ["INVOICE_AI_INFER_CONFIG"] = "configs/train_colab.yaml"
    !python -m uvicorn invoice_ai.api.app_combined:app --host 0.0.0.0 --port 7860
else:
    print("Set RUN_SERVER=True to launch the API + demo UI.")


## Final checklists

**Deployment**
- [ ] `artifacts/models/layout_extractor/latest/` has weights + `labels.json` + `model_metadata.json`
- [ ] `invoice-ai evaluate` shows entity-F1; the agent correctly **flags** a non-reconciling invoice
- [ ] API `/health` returns `models_loaded.layout_extractor = true`

**Submission**
- [ ] `report.pdf` · `slides.pptx` · `submission_bundle.zip` · `grading_checklist.json` (all PASS)
- [ ] Push the repo (models/ + artifacts/ git-ignored) to your public GitHub repo

> ⚠️ `microsoft/layoutlmv3-base` is non-commercial (CC-BY-NC-SA). For a commercial build, set `LAYOUT_MODEL = SCUT-DLVCLab/lilt-roberta-en-base` (MIT).
